# ToneHound • MERT training in Kaggle

Select a GPU accelerator and enable Internet for installing dependencies and reading the public code/model repositories. Attach your **private prepared ToneHound dataset**. Keep its title slug in `DATASET_SLUG` below. Download/save notebook output to preserve the experiment when the session ends. Dataset files are read-only under `/kaggle/input`.

[Full walkthrough](https://github.com/SyhmZlkrn/ToneHound/blob/main/docs/CLOUD_TRAINING.md)


In [ ]:
from pathlib import Path
import subprocess, sys
WORK = Path("/kaggle/working")
DATASET_SLUG = "your-private-tonehound-dataset"
ML_ROOT = Path("/kaggle/input") / DATASET_SLUG
OUTPUT_ROOT = WORK / "experiments"
REPO = WORK / "ToneHound"
CODE_REF = "main"
REMOTE = "https://github.com/SyhmZlkrn/ToneHound.git"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", REMOTE, str(REPO)], check=True)
else:
    assert subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip() == REMOTE
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", CODE_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-training.txt")], check=True)
import torch
assert torch.cuda.is_available(), "Select a GPU accelerator"
print(torch.__version__, torch.cuda.get_device_name())


## Choose a dataset and experiment

Start with a small, reviewed collection (4 full rigs and 4 independent DI performances). The 500-NAM collection is not included. Copy `configs/dataset_manifest.example.json` beside your `nam/` and `di/` folders, replace the entries and fill in their actual cabinet/permission evidence. Use the same performance group for edits of one recording. With 20 recordings, a reasonable first split is 14 train / 3 validation / 3 test.

`SMOKE_RUN` checks the plumbing with one optimizer step; it still embeds the full manifest for the baseline. Use a small manifest first. A smoke result is not an accuracy measurement.


In [ ]:
CONFIG = "configs/mert_amp_v1.yaml"  # Later: mert_amp_lora.yaml / mert_amp_unfreeze_last.yaml
RUN_NAME = "mert_amp_v1_smoke"        # Use a NEW name for each config/dataset
SMOKE_RUN = True
SOURCE_MANIFEST = ML_ROOT / "source" / "manifest.json"
DATASET = ML_ROOT / "datasets" / "amp-tone-v1"
RUN = OUTPUT_ROOT / RUN_NAME


## Train, or resume the same run

This saves to persistent output storage. A frozen baseline is fitted first, using only training and validation data. LoRA and partial fine-tuning update MERT; the frozen configuration updates only the small retrieval head. The active desktop matcher is never changed.


In [ ]:
import yaml
cfg = yaml.safe_load((REPO / CONFIG).read_text())
if SMOKE_RUN:
    cfg["training"].update(epochs=1, steps_per_epoch=1, classes_per_batch=2, eval_batch_size=1, checkpoint_every_steps=1)
run_config = WORK / "tonehound-run-config.yaml"
run_config.write_text(yaml.safe_dump(cfg, sort_keys=False))
command = [sys.executable, "engine/training/train_mert.py", "--config", str(run_config), "--dataset", str(DATASET), "--output", str(RUN)]
if (RUN / "checkpoint" / "last.pt").exists():
    command.append("--resume")
subprocess.run(command, cwd=REPO, check=True)


## Inspect validation before opening the test set

For a real comparison, complete the planned experiment configurations on the **same dataset**, then select using validation only. Do not repeatedly change settings after looking at the test score. The example selection command is in `docs/CLOUD_TRAINING.md`.


In [ ]:
import json
print(json.dumps(json.loads((RUN / "metrics.json").read_text()), indent=2))


## Final held-out evaluation (run only after selecting the experiment)

A separate cell makes test access deliberate. Compare against the frozen encoder/projection on the same dataset. When the selected run uses MERT-95M, set `REFERENCE_RUN` to the completed `mert_amp_v1` experiment to also compare with the current 330M encoder's projection method. Old 51-capture percentages are not comparable to a new 500-capture test.


In [ ]:
REFERENCE_RUN = None  # e.g. OUTPUT_ROOT / "mert_amp_v1"
command = [sys.executable, "engine/training/evaluate_mert.py", "--run", str(RUN), "--dataset", str(DATASET)]
if REFERENCE_RUN is not None:
    command += ["--reference-run", str(REFERENCE_RUN)]
subprocess.run(command, cwd=REPO, check=True)


## Collect the results

Keep the **whole experiment folder**: config, checkpoint, metrics, training history, retrieval test, confusion counts and run information. `checkpoint/best.safetensors` contains the head and any trained MERT updates; it requires the pinned base model and is not a drop-in replacement for the desktop app's `active.npz`. Bring this folder back for local evaluation/integration. Keep audio, checkpoints, API keys and notebook outputs out of the source repository.


In [ ]:
for path in sorted(RUN.rglob("*")):
    if path.is_file():
        print(path.relative_to(RUN), f"{path.stat().st_size / 1024:.1f} KiB")
